In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

# Global plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

2 Load and Parse Dataset

In [ ]:
print("=" * 60)
print("SECTION 2: Loading Dataset")
print("=" * 60)
df = pd.read_csv(
    '/kaggle/input/datasets/pratikhiremath/cancer-clinical-trial1/labeledEligibilitySample1000000.csv',
    sep='\t',
    engine='python',
    on_bad_lines='skip',
    header=None,
    names=['raw']
)
 
print(f"Raw rows loaded : {len(df):,}")
print(f"Sample raw row  : {df['raw'].iloc[0]}")

3 Extract Structured Columns

In [ ]:
print("\n" + "=" * 60)
print("SECTION 3: Extracting Structured Columns")
print("=" * 60)
 
# Extract DRUG
df['drug'] = df['raw'].str.extract(
    r'study interventions are (.+?)\s*\.', flags=re.IGNORECASE
)
 
# Extract CANCER TYPE
df['cancer_type'] = df['raw'].str.extract(
    r'\.\s*(.+?)\s+diagnosis and', flags=re.IGNORECASE
)
 
# Extract ELIGIBILITY CRITERIA TEXT
df['criteria_text'] = df['raw'].str.extract(
    r'diagnosis and\s+(.+)', flags=re.IGNORECASE
)
 
# Drop unparseable rows
df = df.dropna(subset=['drug', 'cancer_type', 'criteria_text']).reset_index(drop=True)
 
print(f"Rows after extraction : {len(df):,}")
print(f"Columns               : {df.columns.tolist()}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nSample:\n{df[['drug','cancer_type','criteria_text']].head(3)}")

4 Text Cleaning

In [ ]:
print("\n" + "=" * 60)
print("SECTION 4: Cleaning Criteria Text")
print("=" * 60)
 
def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower().strip()
    text = re.sub(r'\d+\.\s*', '', text)             # remove leading numbers
    text = re.sub(r'[^\w\s\-\≥\≤\>\<\/\.]', '', text) # keep medical symbols
    text = re.sub(r'\s+', ' ', text)
    return text
 
df['clean_criteria'] = df['criteria_text'].apply(clean_text)
df['clean_cancer']   = df['cancer_type'].str.lower().str.strip()
df['clean_drug']     = df['drug'].str.lower().str.strip()
df['criteria_length'] = df['clean_criteria'].apply(len)
 
print("Text cleaning complete.")
print(f"Avg criteria length : {df['criteria_length'].mean():.0f} characters")
print(f"\nCleaned sample:\n{df['clean_criteria'].head(3).values}")
 

5 Exploratory Data Analysis(EDA)

In [ ]:

print("\n" + "=" * 60)
print("SECTION 5: Exploratory Data Analysis")
print("=" * 60)
 
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Section 5 — Dataset Overview', fontsize=14, fontweight='bold')
 
# 5a. Top 15 cancer types
top_cancers = df['clean_cancer'].value_counts().head(15)
top_cancers.sort_values().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 15 Cancer Types')
axes[0].set_xlabel('Count')
 
# 5b. Top 15 drugs
top_drugs = df['clean_drug'].value_counts().head(15)
top_drugs.sort_values().plot(kind='barh', ax=axes[1], color='mediumseagreen')
axes[1].set_title('Top 15 Drugs / Interventions')
axes[1].set_xlabel('Count')
 
# 5c. Criteria text length distribution
axes[2].hist(df['criteria_length'], bins=50, color='coral', edgecolor='white')
axes[2].set_title('Criteria Text Length Distribution')
axes[2].set_xlabel('Character Length')
axes[2].set_ylabel('Frequency')
 
plt.tight_layout()
plt.savefig('eda_overview.png', bbox_inches='tight')
plt.show()
 
print(f"\nTop 5 Cancer Types:\n{top_cancers.head()}")
print(f"\nTop 5 Drugs:\n{top_drugs.head()}")

6 WordCloud

In [ ]:
print("\n" + "=" * 60)
print("SECTION 6: WordCloud of Eligibility Criteria")
print("=" * 60)
 
all_text = ' '.join(df['clean_criteria'].dropna())
 
wc = WordCloud(
    width=1200, height=500,
    background_color='white',
    colormap='RdYlBu',
    max_words=150
).generate(all_text)
 
plt.figure(figsize=(14, 6))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Most Common Words in Eligibility Criteria', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('wordcloud_criteria.png', bbox_inches='tight')
plt.show()

7 Age_Bias Detection

In [ ]:
# Word to Number Converter + New Extractors 
# 7.1 Map number words to digits
number_words = {
    'zero':0,'one':1,'two':2,'three':3,'four':4,'five':5,
    'six':6,'seven':7,'eight':8,'nine':9,'ten':10,
    'eleven':11,'twelve':12,'thirteen':13,'fourteen':14,'fifteen':15,
    'sixteen':16,'seventeen':17,'eighteen':18,'nineteen':19,'twenty':20,
    'twenty-one':21,'twenty-two':22,'twenty-three':23,'twenty-four':24,
    'twenty-five':25,'twenty-six':26,'twenty-seven':27,'twenty-eight':28,
    'twenty-nine':29,'thirty':30,'thirty-five':35,'forty':40,
    'forty-five':45,'fifty':50,'fifty-five':55,'sixty':60,
    'sixty-five':65,'seventy':70,'seventy-five':75,'eighty':80,
    'eighty-five':85,'ninety':90,'ninety-five':95,'hundred':100
}

def words_to_num(text):
    """Replace number words with digits in a string"""
    if pd.isna(text):
        return text
    result = text.lower()
    # replace longest phrases first to avoid partial replacements
    for word, num in sorted(number_words.items(), key=lambda x: -len(x[0])):
        result = re.sub(rf'\b{word}\b', str(num), result)
    return result

# 7.2: Apply word conversion to clean criteria
df['criteria_numeric'] = df['clean_criteria'].apply(words_to_num)

# Verify conversion worked
print("=== Before conversion ===")
print(df['clean_criteria'].iloc[4])
print("\n=== After conversion ===")
print(df['criteria_numeric'].iloc[4])

In [ ]:
# 7.3 Extract age from criteria_numeric column
def extract_age_v2(text):
    if pd.isna(text):
        return None
    patterns = [
        r'age\s+(?:greater_than|less_than|equal_than|greater_than_or_equal|less_than_or_equal)\s+(\d+)',
        r'at least\s+(\d+)\s+(?:years?|yrs?)',
        r'(\d+)\s+years?\s+(?:of age\s+)?or\s+older',
        r'(?:older|younger)\s+than\s+(\d+)',
        r'(?:minimum|maximum)\s+age\s+(?:of\s+)?(\d+)',
        r'\bage[d]?\s+(\d+)',
        r'(\d+)\s+years?\s+of\s+age',
    ]
    for p in patterns:
        m = re.search(p, text, re.IGNORECASE)
        if m:
            age = int(m.group(1))
            if 10 <= age <= 100:
                return age
    return None

df['extracted_age'] = df['criteria_numeric'].apply(extract_age_v2)
age_df = df[df['extracted_age'].notna()].copy()
age_df['extracted_age'] = age_df['extracted_age'].astype(float)

print(f"Total rows with age threshold : {len(age_df):,}")
print(f"Average trial age cutoff      : {age_df['extracted_age'].mean():.1f}")
print(f"Real-world avg cancer age     : 66.0")
print(f"Gap                           : {66 - age_df['extracted_age'].mean():.1f} years")
print(f"\nAge distribution:\n{age_df['extracted_age'].describe()}")

In [ ]:
# 7.4: Age bucket classification---
bins   = [0, 18, 40, 60, 65, 70, 100]
labels = ['<18', '18–40', '41–60', '61–65', '66–70', '>70']
age_df['age_bucket'] = pd.cut(age_df['extracted_age'],
                               bins=bins, labels=labels)
bucket_counts = age_df['age_bucket'].value_counts().sort_index()

print("Age Bucket Distribution:")
print(bucket_counts)

In [ ]:
# 7.5: Age by cancer type
age_by_cancer = (
    age_df.groupby('clean_cancer')['extracted_age']
    .agg(['mean', 'count'])
    .query('count >= 5')
    .sort_values('mean')
    .head(15)['mean']
    .astype(float)
)

print("Most Age-Restrictive Cancer Types:")
print(age_by_cancer)

In [ ]:
# 7.6:Charts
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('Age Bias Detection in Cancer Clinical Trials',
             fontsize=14, fontweight='bold')

# 7.6.1— Distribution
axes[0].hist(age_df['extracted_age'], bins=30,
             color='coral', edgecolor='white', alpha=0.85)
axes[0].axvline(66, color='darkred', linestyle='--', linewidth=2,
                label='Real avg cancer age (66)')
axes[0].axvline(age_df['extracted_age'].mean(), color='navy',
                linestyle=':', linewidth=2,
                label=f"Trial avg cutoff ({age_df['extracted_age'].mean():.0f})")
axes[0].set_title('Age Threshold Distribution')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Number of Criteria')
axes[0].legend()

# 7.6.2— Buckets
colors = ['Red','Blue','Green','Violet','Yellow','Orange']
axes[1].bar(bucket_counts.index, bucket_counts.values,
            color=colors, edgecolor='white')
axes[1].set_title('Age Cutoff Buckets')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Number of Criteria')
for i, v in enumerate(bucket_counts.values):
    axes[1].text(i, v + 20, str(v), ha='center', fontsize=9)

# 7.6.3 — By cancer type
if len(age_by_cancer) > 0:
    age_by_cancer.plot(kind='barh', ax=axes[2],
                       color='salmon', edgecolor='white')
    axes[2].axvline(66, color='darkred', linestyle='--',
                    linewidth=1.5, label='Real avg age (66)')
    axes[2].set_title('Most Age-Restrictive Cancer Types')
    axes[2].set_xlabel('Average Age Threshold')
    axes[2].legend(fontsize=8)
else:
    axes[2].text(0.5, 0.5, 'Not enough grouped data',
                 ha='center', va='center', transform=axes[2].transAxes)

plt.tight_layout()
plt.savefig('age_bias_detection.png', bbox_inches='tight')
plt.show()

8 Prior Treatment Penalty 

In [ ]:
print("\n" + "=" * 60)
print("SECTION 8: Prior Treatment Penalty Analysis")
print("=" * 60)
 
treatment_patterns = {
    'Prior chemotherapy'   : r'prior\s+chemo(?:therapy)?',
    'Prior radiation'      : r'prior\s+radi(?:ation|otherapy)',
    'Prior immunotherapy'  : r'prior\s+immuno(?:therapy)?',
    'Treatment naive'      : r'treatment[- ]naive',
    'No prior treatment'   : r'no\s+prior\s+(?:anti-?cancer\s+)?treatment',
    'Must not have received': r'must not have received',
    'No previous systemic' : r'no\s+previous\s+systemic',
    'No prior surgery'     : r'no\s+prior\s+surgery',
}
 
counts = {}
for label, pattern in treatment_patterns.items():
    n = df['clean_criteria'].str.contains(pattern, na=False, regex=True).sum()
    counts[label] = n
 
counts_series = pd.Series(counts).sort_values(ascending=True)
pct = (counts_series / len(df) * 100).round(2)
 
print("Prior Treatment Exclusion Breakdown:")
for k, v in counts_series.items():
    print(f"  {k:<30} {v:>7,}  ({pct[k]:.2f}%)")
 
plt.figure(figsize=(10, 5))
counts_series.plot(kind='barh', color='tomato', edgecolor='white')
plt.title('Section 8 — Prior Treatment Exclusion Patterns', fontsize=13, fontweight='bold')
plt.xlabel('Number of Criteria Containing Pattern')
for i, (val, pct_val) in enumerate(zip(counts_series, pct)):
    plt.text(val + 50, i, f'{pct_val:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('prior_treatment.png', bbox_inches='tight')
plt.show()

9 Eligibility Barrier Score

In [ ]:
print("\n" + "=" * 60)
print("SECTION 9: Eligibility Barrier Score (EBS) — Original Metric")
print("=" * 60)
 
def compute_ebs(text):
    """
    Eligibility Barrier Score (EBS) — range 0 to 100.
    Measures how restrictive a trial's eligibility criteria is.
    Higher = harder for a real world patient to qualify.
 
    Scoring:
      Age restriction found        → +20 pts
      Prior treatment exclusion    → +25 pts
      Lab value threshold          → +20 pts
      Performance status (ECOG)    → +15 pts
      Gender restriction           → +10 pts
      Comorbidity exclusion        → +10 pts
    """
    if pd.isna(text):
        return 0
    score = 0
    if re.search(r'age\s*[≥≤<>]|\b(?:older|younger) than\b', text, re.I):
        score += 20
    if re.search(r'prior\s+(?:chemo|radiation|treatment|immuno)|treatment[- ]naive|must not have received', text, re.I):
        score += 25
    if re.search(r'\b(?:hemoglobin|platelet|creatinine|bilirubin|alt|ast|wbc|anc|egfr)\b', text, re.I):
        score += 20
    if re.search(r'\becog\b|performance status|karnofsky', text, re.I):
        score += 15
    if re.search(r'\b(?:male|female|women|men)\s+only\b', text, re.I):
        score += 10
    if re.search(r'(?:diabetes|hypertension|cardiac|renal|hepatic)\s+(?:disease|failure|dysfunction)', text, re.I):
        score += 10
    return score
 
df['EBS'] = df['clean_criteria'].apply(compute_ebs)
 
print(f"Average EBS            : {df['EBS'].mean():.1f} / 100")
print(f"Highly restrictive (>60): {(df['EBS'] > 60).sum():,} ({(df['EBS']>60).mean()*100:.1f}%)")
print(f"Open access (EBS = 0)  : {(df['EBS'] == 0).sum():,} ({(df['EBS']==0).mean()*100:.1f}%)")
print(f"\nEBS Distribution:\n{df['EBS'].value_counts().sort_index()}")
 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Section 9 — Eligibility Barrier Score (EBS)', fontsize=14, fontweight='bold')
 
# EBS distribution
axes[0].hist(df['EBS'], bins=20, color='purple', edgecolor='white', alpha=0.8)
axes[0].axvline(df['EBS'].mean(), color='gold', linestyle='--', linewidth=2,
                label=f"Mean EBS = {df['EBS'].mean():.1f}")
axes[0].set_title('EBS Distribution Across All Trials')
axes[0].set_xlabel('Eligibility Barrier Score (0=Open, 100=Highly Restrictive)')
axes[0].set_ylabel('Count')
axes[0].legend()
 
# Average EBS by cancer type
ebs_by_cancer = (
    df.groupby('clean_cancer')['EBS']
    .mean()
    .sort_values(ascending=False)
    .head(15)
)
ebs_by_cancer.sort_values().plot(kind='barh', ax=axes[1], color='mediumpurple')
axes[1].set_title('Top 15 Most Restrictive Cancer Types (by EBS)')
axes[1].set_xlabel('Average EBS')
 
plt.tight_layout()
plt.savefig('ebs_analysis.png', bbox_inches='tight')
plt.show()

10 Patient Reachability Index (PRI)

In [ ]:
print("\n" + "=" * 60)
print("SECTION 10: Patient Reachability Index (PRI) — Original Metric")
print("=" * 60)
 
def compute_pri(text):
    """
    Patient Reachability Index (PRI) — starts at 100%, shrinks per barrier.
    Estimates what % of real cancer patients could qualify for this trial.
 
    Based on real-world cancer population statistics:
      Age cutoff ≥ 65         → ×0.60  (40% of patients are under 65)
      Prior treatment excl.   → ×0.35  (65% of patients had prior treatment)
      ECOG ≤ 1 required       → ×0.55  (45% have ECOG > 1)
      Organ dysfunction excl. → ×0.80
      Treatment naive only    → ×0.30
      Gender restricted       → ×0.50
    """
    if pd.isna(text):
        return 100.0
    reachability = 1.0
    if re.search(r'age\s*[≥>]\s*(?:6[5-9]|[7-9]\d)', text, re.I):
        reachability *= 0.60
    if re.search(r'prior\s+(?:chemo|radiation|treatment)|must not have received', text, re.I):
        reachability *= 0.35
    if re.search(r'ecog\s*(?:score\s*)?[≤<]\s*[01]|performance status\s*[≤<]\s*[01]', text, re.I):
        reachability *= 0.55
    if re.search(r'(?:renal|hepatic|cardiac)\s+(?:failure|disease|dysfunction)', text, re.I):
        reachability *= 0.80
    if re.search(r'treatment[- ]naive', text, re.I):
        reachability *= 0.30
    if re.search(r'\b(?:male|female|women|men)\s+only\b', text, re.I):
        reachability *= 0.50
    return round(reachability * 100, 1)
 
df['PRI'] = df['clean_criteria'].apply(compute_pri)
 
print(f"Average PRI                        : {df['PRI'].mean():.1f}%")
print(f"Trials reachable by < 25% patients : {(df['PRI'] < 25).sum():,} ({(df['PRI']<25).mean()*100:.1f}%)")
print(f"Trials reachable by > 75% patients : {(df['PRI'] > 75).sum():,} ({(df['PRI']>75).mean()*100:.1f}%)")
 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Section 10 — Patient Reachability Index (PRI)', fontsize=14, fontweight='bold')
 
# PRI distribution
axes[0].hist(df['PRI'], bins=25, color='teal', edgecolor='white', alpha=0.8)
axes[0].axvline(df['PRI'].mean(), color='red', linestyle='--', linewidth=2,
                label=f"Mean PRI = {df['PRI'].mean():.1f}%")
axes[0].set_title('PRI Distribution — % of Real Patients Who Can Qualify')
axes[0].set_xlabel('Patient Reachability Index (%)')
axes[0].set_ylabel('Count')
axes[0].legend()
 
# PRI by cancer type (most exclusive)
pri_by_cancer = (
    df.groupby('clean_cancer')['PRI']
    .mean()
    .sort_values(ascending=True)
    .head(15)
)
pri_by_cancer.plot(kind='barh', ax=axes[1], color='cadetblue')
axes[1].set_title('Top 15 Most Exclusive Cancer Types (Lowest PRI)')
axes[1].set_xlabel('Average PRI (%)')
 
plt.tight_layout()
plt.savefig('pri_analysis.png', bbox_inches='tight')
plt.show()

11 EBS VS PRI Corelation

In [ ]:
print("\n" + "=" * 60)
print("SECTION 11: EBS vs PRI Relationship")
print("=" * 60)
 
corr = df[['EBS', 'PRI']].corr().loc['EBS', 'PRI']
print(f"Pearson Correlation (EBS vs PRI): {corr:.3f}")
 
plt.figure(figsize=(8, 5))
sample = df.sample(min(5000, len(df)), random_state=42)
plt.scatter(sample['EBS'], sample['PRI'], alpha=0.15, color='slateblue', s=10)
plt.xlabel('Eligibility Barrier Score (EBS)')
plt.ylabel('Patient Reachability Index (PRI %)')
plt.title(f'Section 11 — EBS vs PRI  (r = {corr:.2f})', fontweight='bold')
plt.tight_layout()
plt.savefig('ebs_vs_pri.png', bbox_inches='tight')
plt.show()

12 Final Summary Dashboard

In [ ]:
print("\n" + "=" * 60)
print("SECTION 12: Final Summary Dashboard")
print("=" * 60)
 
fig = plt.figure(figsize=(18, 12))
fig.suptitle(
    'CancerGate: Measuring Hidden Barriers in Cancer Clinical Trial Eligibility\n'
    'Eligibility Barrier Score (EBS) & Patient Reachability Index (PRI)',
    fontsize=14, fontweight='bold', y=1.01
)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
 
# 1. EBS Distribution
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(df['EBS'], bins=20, color='mediumpurple', edgecolor='white', alpha=0.85)
ax1.axvline(df['EBS'].mean(), color='gold', linestyle='--',
            label=f"Mean={df['EBS'].mean():.0f}")
ax1.set_title('Eligibility Barrier Score', fontweight='bold')
ax1.set_xlabel('EBS (0=Open, 100=Restrictive)')
ax1.legend(fontsize=8)
 
# 2. PRI Distribution
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(df['PRI'], bins=25, color='cadetblue', edgecolor='white', alpha=0.85)
ax2.axvline(df['PRI'].mean(), color='red', linestyle='--',
            label=f"Mean={df['PRI'].mean():.1f}%")
ax2.set_title('Patient Reachability Index', fontweight='bold')
ax2.set_xlabel('PRI (% of patients who qualify)')
ax2.legend(fontsize=8)
 
# 3. Age Bias
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(age_df['extracted_age'], bins=25, color='coral', edgecolor='white', alpha=0.85)
ax3.axvline(66, color='darkred', linestyle='--', label='Real avg age (66)')
ax3.set_title('Age Threshold Bias', fontweight='bold')
ax3.set_xlabel('Age Cutoff in Trial')
ax3.legend(fontsize=8)
 
# 4. Prior Treatment Exclusions
ax4 = fig.add_subplot(gs[1, 0])
counts_series.plot(kind='barh', ax=ax4, color='tomato')
ax4.set_title('Prior Treatment Exclusions', fontweight='bold')
ax4.set_xlabel('Count')
ax4.tick_params(axis='y', labelsize=7)
 
# 5. Top Cancer Types
ax5 = fig.add_subplot(gs[1, 1])
top_cancers.head(10).sort_values().plot(kind='barh', ax=ax5, color='steelblue')
ax5.set_title('Top 10 Cancer Types', fontweight='bold')
ax5.tick_params(axis='y', labelsize=7)
 
# 6. Key Stats Summary
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
stats = [
    f"Total criteria analysed : {len(df):,}",
    f"Unique cancer types     : {df['clean_cancer'].nunique():,}",
    f"Unique drugs            : {df['clean_drug'].nunique():,}",
    f"",
    f"Eligibility Barrier Score",
    f"Mean EBS                : {df['EBS'].mean():.1f} / 100",
    f"Highly restrictive (>60): {(df['EBS']>60).mean()*100:.1f}%",
    f"",
    f" Patient Reachability Index",
    f"Mean PRI                : {df['PRI'].mean():.1f}%",
    f"PRI < 25% (exclusive)   : {(df['PRI']<25).mean()*100:.1f}%",
    f"",
    f"Age Bias",
    f"Criteria with age limit : {len(age_df):,}",
    f"Avg trial age cutoff    : {age_df['extracted_age'].mean():.0f} yrs",
    f"Real avg cancer age     : 66 yrs",
]
ax6.text(0.05, 0.95, '\n'.join(stats), transform=ax6.transAxes,
         fontsize=9, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax6.set_title('Key Findings', fontweight='bold')
 
plt.savefig('cancergate_dashboard.png', bbox_inches='tight', dpi=150)
plt.show()
print("Dashboard saved as cancergate_dashboard.png")

In [ ]:
13 Conclusion

In [ ]:
print("\n" + "=" * 60)
print("SECTION 13: Key Findings & Conclusion")
print("=" * 60)
 
print(f"""

        CANCERGATE — FINAL FINDINGS SUMMARY             
                                                          
  Total criteria analysed    : {len(df):>10,}               
  Unique cancer types        : {df['clean_cancer'].nunique():>10,}               
  Unique drugs / interventions: {df['clean_drug'].nunique():>9,}               
                                                          
  ── Eligibility Barrier Score (EBS) ──                   
  Average EBS                : {df['EBS'].mean():>9.1f}               
  Highly restrictive (EBS>60): {(df['EBS']>60).mean()*100:>9.1f}%              
                                                          
  ── Patient Reachability Index (PRI) ──                  
  Average PRI                : {df['PRI'].mean():>9.1f}%              
  Trials <25% reachable      : {(df['PRI']<25).mean()*100:>9.1f}%              
                                                          
  ── Age Bias ──                                          
  Avg trial age cutoff       : {age_df['extracted_age'].mean():>9.1f} yrs           
  Real-world avg cancer age  :       66.0 yrs           
  Gap                        : {66 - age_df['extracted_age'].mean():>+9.1f} yrs           
                                                          

 
CONCLUSION:
  CancerGate reveals that the average cancer clinical trial
  is accessible to only {df['PRI'].mean():.0f}% of real-world patients.
  Age bias, prior treatment penalties, and complex lab-value
  requirements systematically exclude the very patients these
  trials are meant to help.
 
  The Eligibility Barrier Score (EBS) and Patient Reachability
  Index (PRI) introduced here offer a replicable, data driven
  framework for evaluating trial accessibility a tool that
  regulators, sponsors, and patient advocates can adopt to
  design more inclusive cancer research.
""")